In [ ]:
!pip install google-genai
!pip install langchain
!pip install langchain-community
!pip install langchain-text-splitters
!pip install pypdf
!pip install scikit-learn

In [ ]:
# imports 

from google import genai

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from tools import get_current_time

In [7]:
# api key 

client = genai.Client(
    api_key="AQ.Ab8RN6LI087wssJ148VI7PNo8z-NJB_MI7gcNb3xEwLDWmVk6A"
)

print("Gemini Connected")

Gemini Connected


In [8]:
# load pdf 

loader = PyPDFLoader(
    "data/python_tutorial.pdf"
)

docs = loader.load()

print("Pages:", len(docs))

Pages: 32


In [9]:
# chunk documents 

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(docs)

print("Chunks:", len(chunks))

Chunks: 180


In [10]:
# Create TF-IDF Vector Store

chunk_texts = [
    chunk.page_content
    for chunk in chunks
]

vectorizer = TfidfVectorizer(
    stop_words="english"
)

vectors = vectorizer.fit_transform(
    chunk_texts
)

print("Vector Store Created")

Vector Store Created


In [11]:
# retrieval function 

def retrieve_context(
    query,
    k=3
):

    query_vector = vectorizer.transform(
        [query]
    )

    similarities = cosine_similarity(
        query_vector,
        vectors
    )[0]

    top_indices = (
        similarities
        .argsort()[-k:][::-1]
    )

    context = "\n\n".join(
        [
            chunk_texts[i]
            for i in top_indices
        ]
    )

    return context

In [ ]:
# test retrieval 

context = retrieve_context(
    "What is Python?"
)

print(context[:1000])

In [13]:
# conversation memory 

chat_history = []

In [14]:
# rewriting questions 

def rewrite_question(
    question,
    history
):

    history_text = "\n".join(
        [
            f"{msg['role']}: {msg['content']}"
            for msg in history
        ]
    )

    prompt = f"""
Convert follow-up questions into standalone questions.

Conversation:
{history_text}

Current Question:
{question}

Return only the standalone question.
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text.strip()

In [15]:
# rag answer function 

def rag_answer(question):

    context = retrieve_context(question)

    prompt = f"""
You are a RAG assistant.

Answer ONLY using the context.

Context:
{context}

Question:
{question}

If answer is unavailable,
say:
'I could not find this in the document.'
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text

In [16]:
# tool definition 

tools = {
    "get_current_time": get_current_time
}

In [17]:
# tool calling function 

def tool_calling(question):

    prompt = f"""
You are a tool selector.

Available Tool:
get_current_time

Question:
{question}

If current time is required,
return exactly:

get_current_time

Otherwise answer normally.
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    decision = response.text.strip()

    if decision == "get_current_time":

        result = get_current_time()

        final_prompt = f"""
User Question:
{question}

Tool Result:
{result}

Generate a natural language answer.
"""

        final_response = (
            client.models.generate_content(
                model="gemini-2.5-flash",
                contents=final_prompt
            )
        )

        return final_response.text

    return decision

In [18]:
# routing logic 

def route_question(question):

    context = retrieve_context(question)

    prompt = f"""
Determine whether the answer
exists in the retrieved context.

Question:
{question}

Context:
{context}

Reply ONLY:

RAG

or

TOOL
"""

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    route = response.text.strip()

    if route == "RAG":
        return rag_answer(question)

    return tool_calling(question)

In [ ]:
# main conversational loop 

while True:

    user_query = input(
        "\nYou: "
    )

    if user_query.lower() == "exit":
        break

    standalone_question = (
        rewrite_question(
            user_query,
            chat_history
        )
    )

    print(
        "\nStandalone Question:"
    )

    print(
        standalone_question
    )

    answer = route_question(
        standalone_question
    )

    print(
        "\nAssistant:"
    )

    print(answer)

    chat_history.append(
        {
            "role":"user",
            "content":user_query
        }
    )

    chat_history.append(
        {
            "role":"assistant",
            "content":answer
        }
    )